<a href="https://colab.research.google.com/github/27Kushal/uncertainty-aware-cnn/blob/main/uncertainity_aware_cnn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader,Subset,Dataset
import torchvision
import torchvision.transforms as transforms
import torchvision.models as models
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve, precision_recall_curve, auc, accuracy_score, f1_score

In [2]:
SEED=42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

Using device: cuda


using CIFAR 10 dataset for the image classification problem so we will use the first 5 types i.e. airplane, automobile, bird, cat, deer. Rest 5 are Out of distribution input.

In [3]:
KNOWN_CLASSES = [0, 1, 2, 3, 4] # airplane, automobile, bird, cat, deer
UNKNOWN_CLASSES = [5, 6, 7, 8, 9]  # dog, frog, horse, ship, truck
KNOWN_NAMES = ['airplane', 'automobile', 'bird', 'cat', 'deer']
ALL_NAMES = ['airplane', 'automobile', 'bird', 'cat', 'deer','dog', 'frog', 'horse', 'ship', 'truck']

# Standard normalization values for CIFAR-10
MEAN = (0.4914, 0.4822, 0.4465)
STD = (0.2023, 0.1994, 0.2010)

In [4]:
#preprocessing for training (including data augmentation)
train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD)
])

#preprocessing for testing(no augmentation)
test_transform=transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD)
])

In [5]:
#filtering CIFAR-10 into known and unknown sets

class SplitCIFAR10(Dataset):
  def __init__(self, base_dataset, target_classes, is_known=True):
    self.base_dataset = base_dataset
    self.is_known = is_known
    #finding indices belonging to target class
    self.indices = [
      i for i, label in enumerate(base_dataset.targets)
      if label in target_classes
    ]
    #mapping known labels to 0,1,2,3,4
    self.label_map = {c: idx for idx, c in enumerate(target_classes)}

  def __len__(self):
    return len(self.indices)

  def __getitem__(self,idx):
    real_idx = self.indices[idx]
    image, orig_label = self.base_dataset[real_idx]
    # Known classes get labels 0 to 4, Unknown classes get -1
    target_label = self.label_map[orig_label] if self.is_known else -1
    return image, target_label, orig_label

In [ ]:
#downloading dataset
raw_train = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=train_transform)
raw_test = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=test_transform)

#filtering the dataset
known_train_set = SplitCIFAR10(raw_train, KNOWN_CLASSES, is_known=True)
known_test_full = SplitCIFAR10(raw_test, KNOWN_CLASSES, is_known=True)
unknown_test_set = SplitCIFAR10(raw_test, UNKNOWN_CLASSES, is_known=False)
#spliting the known test to validation and test sets
known_val_set, known_test_set = random_split(
    known_test_full, [2500, 2500], generator=torch.Generator().manual_seed(SEED)
)

#creating dataloaders
BATCH_SIZE = 128
train_loader = DataLoader(known_train_set, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(known_val_set, batch_size=BATCH_SIZE, shuffle=False)
id_loader    = DataLoader(known_test_set, batch_size=BATCH_SIZE, shuffle=False)  #in distribution input
ood_loader   = DataLoader(unknown_test_set, batch_size=BATCH_SIZE, shuffle=False) #out of distribution input

print(f"Known Training Images   : {len(known_train_set)} (Classes: {KNOWN_NAMES})")
print(f"Known Validation Images : {len(known_val_set)}")
print(f"Known Test Images (ID)  : {len(known_test_set)}")
print(f"Unknown Test Images(OOD): {len(unknown_test_set)} (Excluded from training!)")

 32%|███▏      | 54.3M/170M [28:31<1:22:14, 23.5kB/s]

In [ ]:
#visualizing the data

def show_sample_images(dataset, title, num_images=5):
    fig, axes = plt.subplots(1, num_images, figsize=(12, 2.5))
    fig.suptitle(title, fontsize=14, fontweight='bold')
    for i in range(num_images):
        img_tensor, mapped_label, orig_label = dataset[i]
        # Un-normalize image so colors look normal
        img = img_tensor.numpy().transpose((1, 2, 0))
        img = img * np.array(STD) + np.array(MEAN)
        img = np.clip(img, 0, 1)

        axes[i].imshow(img)
        axes[i].set_title(f"True: {ALL_NAMES[orig_label]}")
        axes[i].axis('off')
    plt.tight_layout()
    plt.show()

show_sample_images(known_test_set, "KNOWN Categories (Model is trained on these 5)")
show_sample_images(unknown_test_set, "UNKNOWN / EXCLUDED Categories (Model has NEVER seen these!)")

In [ ]:
#building the model

class CustomResNet18(nn.Module):
    def __init__(self, num_classes=5, dropout_prob=0.3):
        super().__init__()
        # Load pre-trained ResNet-18
        base = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

        # Tweak 1: Adapt first layer for small 32x32 CIFAR images
        base.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
        base.maxpool = nn.Identity()

        # Keep everything up to the feature extractor
        self.feature_extractor = nn.Sequential(
            base.conv1, base.bn1, base.relu, base.maxpool,
            base.layer1, base.layer2, base.layer3, base.layer4,
            base.avgpool
        )
        # Tweak 2: Add Dropout + Final Linear Layer for our 5 Known classes
        self.dropout = nn.Dropout(p=dropout_prob)
        self.classifier = nn.Linear(512, num_classes)

    def forward(self, x, enable_mc_dropout=False):
        features = self.feature_extractor(x)
        features = torch.flatten(features, 1)
        # If training OR running Monte Carlo Dropout, randomly drop 30% of neurons
        if self.training or enable_mc_dropout:
            features = F.dropout(features, p=self.dropout.p, training=True)
        logits = self.classifier(features)
        return logits

model = CustomResNet18(num_classes=5, dropout_prob=0.3).to(DEVICE)

In [ ]:
#setting up the loss function and optimizer

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

EPOCHS = 12

In [ ]:
# Training Loop
print("Starting Training on the 5 Known Categories...")
for epoch in range(EPOCHS):
    model.train()
    running_loss, correct, total = 0.0, 0, 0

    for images, labels, _ in train_loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)

        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        preds = logits.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    scheduler.step()
    train_acc = (correct / total) * 100

    # Evaluate on Known Validation Set
    model.eval()
    val_correct, val_total = 0, 0
    with torch.inference_mode():
        for images, labels, _ in val_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            logits = model(images)
            val_correct += (logits.argmax(dim=1) == labels).sum().item()
            val_total += labels.size(0)
    val_acc = (val_correct / val_total) * 100

    print(f"Epoch [{epoch+1:02d}/{EPOCHS:02d}] | Loss: {running_loss/total:.4f} | "
          f"Train Acc: {train_acc:.2f}% | Known Val Acc: {val_acc:.2f}%")

In [ ]:
model.eval()
known_preds, known_targets, known_confidences = [], [], []

with torch.inference_mode():
    for images, labels, _ in id_loader:
        images = images.to(DEVICE)
        logits = model(images)
        # converting raw logits into probabilities (0.0 to 1.0) using Softmax
        probs = F.softmax(logits, dim=1)

        # Get the highest probability (confidence) and the predicted class (0..4)
        conf, preds = torch.max(probs, dim=1)

        known_preds.extend(preds.cpu().numpy())
        known_targets.extend(labels.numpy())
        known_confidences.extend(conf.cpu().numpy())

known_preds = np.array(known_preds)
known_targets = np.array(known_targets)
known_confidences = np.array(known_confidences)

print("Evaluation on Unseen KNOWN Categories")
print(classification_report(known_targets, known_preds, target_names=KNOWN_NAMES, digits=4))
print(f"Average Confidence on Known Images: {np.mean(known_confidences)*100:.2f}%\n")

In [ ]:
unknown_preds, unknown_orig_labels, unknown_confidences = [], [], []

with torch.is_inference_mode():
    for images, _, orig_labels in ood_loader:
        images = images.to(DEVICE)
        logits = model(images)
        probs = F.softmax(logits, dim=1)
        conf, preds = torch.max(probs, dim=1)

        unknown_preds.extend(preds.cpu().numpy())
        unknown_orig_labels.extend(orig_labels.numpy())
        unknown_confidences.extend(conf.cpu().numpy())

unknown_preds = np.array(unknown_preds)
unknown_orig_labels = np.array(unknown_orig_labels)
unknown_confidences = np.array(unknown_confidences)

print("When the Model Sees UNKNOWN Categories?")
print(f"Total Unknown Images Tested: {len(unknown_preds)}")
print(f"Average Confidence on UNKNOWN Images: {np.mean(unknown_confidences)*100:.2f}%")
print(f"Percentage of Unknown Images predicted with >80% confidence: "
      f"{np.mean(unknown_confidences > 0.80)*100:.2f}%")
print(f"Percentage of Unknown Images predicted with >90% confidence: "
      f"{np.mean(unknown_confidences > 0.90)*100:.2f}%\n")

In [ ]:
# Plot Confusion Matrix for Known Classes
plt.figure(figsize=(6, 5))
cm = confusion_matrix(known_targets, known_preds)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=KNOWN_NAMES, yticklabels=KNOWN_NAMES)
plt.title("Confusion Matrix on Known Categories")
plt.xlabel("Predicted Category")
plt.ylabel("True Category")
plt.show()

In [ ]:
def get_all_uncertainty_scores(model, dataloader, mc_passes=10):
    """
    Runs the model on a dataloader and computes 4 confidence/uncertainty scores.
    Note: We orient all 4 scores so that:
      HIGHER score = Model is CERTAIN (Likely Known / In-Distribution)
      LOWER score  = Model is UNCERTAIN (Likely Unknown / Out-of-Distribution)
    """
    model.eval()
    data = {
        'preds': [], 'true_known_label': [], 'orig_cifar_label': [],
        'msp': [], 'neg_entropy': [], 'energy_conf': [], 'mc_dropout_conf': []
    }

    with torch.inference_mode():
        for images, target_labels, orig_labels in dataloader:
            images = images.to(DEVICE)
            logits = model(images, enable_mc_dropout=False)
            probs = F.softmax(logits, dim=1)

            # 1. MSP: Maximum Softmax Probability
            msp, preds = torch.max(probs, dim=1)

            # 2. Negative Entropy: - ( -sum(p * log(p)) )
            # Higher entropy means more confused, so we negate it so Higher = More Certain
            entropy = -torch.sum(probs * torch.log(probs + 1e-12), dim=1)
            neg_entropy = -entropy

            # 3. Energy Confidence Score: log( sum( exp(logits) ) )
            # Directly measures raw logit activation strength before Softmax!
            energy_conf = torch.logsumexp(logits, dim=1)

            # 4. Monte Carlo (MC) Dropout Confidence (10 stochastic forward passes)
            mc_probs_sum = torch.zeros_like(probs)
            for _ in range(mc_passes):
                mc_logits = model(images, enable_mc_dropout=True)
                mc_probs_sum += F.softmax(mc_logits, dim=1)
            mc_avg_probs = mc_probs_sum / mc_passes
            mc_conf, _ = torch.max(mc_avg_probs, dim=1)

            # Save everything
            data['preds'].extend(preds.cpu().numpy())
            data['true_known_label'].extend(target_labels.numpy())
            data['orig_cifar_label'].extend(orig_labels.numpy())
            data['msp'].extend(msp.cpu().numpy())
            data['neg_entropy'].extend(neg_entropy.cpu().numpy())
            data['energy_conf'].extend(energy_conf.cpu().numpy())
            data['mc_dropout_conf'].extend(mc_conf.cpu().numpy())

    for k in data:
        data[k] = np.array(data[k])
    return data

In [ ]:
print("Calculating all 4 Uncertainty Scores on Validation, Known Test, and Unknown Test sets...")
val_scores     = get_all_uncertainty_scores(model, val_loader)
known_scores   = get_all_uncertainty_scores(model, id_loader)
unknown_scores = get_all_uncertainty_scores(model, ood_loader)

In [ ]:
# Compare which of the 4 methods is best at separating Known (1) from Unknown (0) using AUROC
y_true_ood = np.concatenate([np.ones(len(known_scores['msp'])), np.zeros(len(unknown_scores['msp']))])

comparison_rows = []
for name, key in [
    ("1. Max Softmax Prob (MSP)", "msp"),
    ("2. Predictive Entropy", "neg_entropy"),
    ("3. Energy Score (Logit-based)", "energy_conf"),
    ("4. Monte Carlo Dropout (10x)", "mc_dropout_conf")
]:
    y_score = np.concatenate([known_scores[key], unknown_scores[key]])
    auroc = roc_auc_score(y_true_ood, y_score) * 100
    comparison_rows.append({"Uncertainty Method": name, "OOD Detection AUROC (%)": round(auroc, 2)})

comp_df = pd.DataFrame(comparison_rows)
print("\nComparison of Uncertainty Estimation Methods (Higher AUROC = Better)")
print(comp_df.to_string(index=False))

In [ ]:
# 1. Pick our threshold using ONLY the Known Validation Set
# np.percentile(..., 10) means 90% of Known validation images have a score higher than this cutoff
BEST_METHOD = 'energy_conf'
THRESHOLD = np.percentile(val_scores[BEST_METHOD], 10)
print(f"Selected Energy Threshold (from Validation Set): {THRESHOLD:.4f}\n")

# 2. Combine Known Test (2,500) + Unknown Test (5,000) = 7,500 Total Open-World Images
all_preds  = np.concatenate([known_scores['preds'], unknown_scores['preds']])
all_true   = np.concatenate([known_scores['true_known_label'], unknown_scores['true_known_label']]) # -1 for Unknown
all_scores = np.concatenate([known_scores[BEST_METHOD], unknown_scores[BEST_METHOD]])

#BEFORE Uncertainty Handling
# The model is forced to guess 0..4 on every image (it can NEVER output -1)
acc_before = np.mean(all_preds == all_true) * 100
unknown_caught_before = 0.0  # 0% of unknown images are rejected

#AFTER Uncertainty Handling
# If score < THRESHOLD, override prediction with -1 ("Uncertain / Unknown")
preds_after = np.where(all_scores < THRESHOLD, -1, all_preds)
acc_after = np.mean(preds_after == all_true) * 100

# How many of the 5,000 Unknown images did we successfully catch as -1?
unknown_preds_after = preds_after[len(known_scores['preds']):]
unknown_caught_after = np.mean(unknown_preds_after == -1) * 100

# Selective Accuracy: When the model IS confident enough to answer (not -1), how often is it right?
confident_mask = (preds_after != -1)
selective_acc_after = np.mean(all_preds[confident_mask] == all_true[confident_mask]) * 100


In [ ]:
# Print a clean comparison table for your report/README
comparison_table = pd.DataFrame({
    "Metric": [
        "Overall Open-World Accuracy (Known + Unknown mix)",
        "Unknown (OOD) Images Successfully Flagged as Uncertain",
        "Accuracy on Images the Model Chooses to Predict"
    ],
    "WITHOUT Uncertainty Handling": [
        f"{acc_before:.2f}%",
        f"{unknown_caught_before:.2f}% (Fails completely)",
        f"{acc_before:.2f}%"
    ],
    "WITH Uncertainty Handling (Energy)": [
        f"{acc_after:.2f}%",
        f"{unknown_caught_after:.2f}%",
        f"{selective_acc_after:.2f}%"
    ]
})

print("Comparison Before vs. After Uncertainty Handling")
print(comparison_table.to_string(index=False))

In [ ]:
#plotting confidence distributions (known vs unknown)

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

# Left Plot: Standard Softmax Confidence (MSP)
sns.histplot(known_scores['msp'], bins=30, color='royalblue', label='Known (ID)', kde=True, stat='density', alpha=0.5, ax=axes[0])
sns.histplot(unknown_scores['msp'], bins=30, color='crimson', label='Unknown (OOD)', kde=True, stat='density', alpha=0.5, ax=axes[0])
axes[0].set_title("1. Standard Softmax Probability (MSP)")
axes[0].set_xlabel("Max Softmax Confidence (0.0 to 1.0)")
axes[0].legend()

# Right Plot: Energy Score with our Cutoff Threshold
sns.histplot(known_scores['energy_conf'], bins=30, color='royalblue', label='Known (ID)', kde=True, stat='density', alpha=0.5, ax=axes[1])
sns.histplot(unknown_scores['energy_conf'], bins=30, color='crimson', label='Unknown (OOD)', kde=True, stat='density', alpha=0.5, ax=axes[1])
axes[1].axvline(THRESHOLD, color='black', linestyle='--', linewidth=2, label=f'Cutoff Threshold ({THRESHOLD:.2f})')
axes[1].set_title("2. Energy Confidence Score (Better Separation!)")
axes[1].set_xlabel("Energy Score (Left of dashed line = Rejected as Unknown)")
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
#finding which unknown class trick the model

ood_analysis_df = pd.DataFrame({
    'True_Unknown_Class': [ALL_NAMES[i] for i in unknown_scores['orig_cifar_label']],
    'Predicted_As_Known': [KNOWN_NAMES[p] for p in unknown_scores['preds']],
    'Energy_Score': unknown_scores['energy_conf'],
    'Tricked_Detector': unknown_scores['energy_conf'] >= THRESHOLD
})

failure_breakdown = ood_analysis_df.groupby('True_Unknown_Class').agg(
    Total_Images=('Tricked_Detector', 'count'),
    Caught_As_Uncertain_Pct=('Tricked_Detector', lambda x: round((1 - np.mean(x)) * 100, 2)),
    Failure_Rate_Pct=('Tricked_Detector', lambda x: round(np.mean(x) * 100, 2)),
    Most_Confused_With=('Predicted_As_Known', lambda x: x.value_counts().index[0])
).reset_index()

print("\n=== Failure Analysis Across the 5 Unknown Categories ===")
print(failure_breakdown.to_string(index=False))

In [ ]:
#visually checking unknown images tricking the detector

failed_indices = np.where(unknown_scores['energy_conf'] >= THRESHOLD)[0]
# Sort by highest energy score (the biggest, most confident blunders!)
worst_failures = failed_indices[np.argsort(unknown_scores['energy_conf'][failed_indices])[::-1][:5]]

fig, axes = plt.subplots(1, 5, figsize=(14, 3))
fig.suptitle("FAILURE CASES: Unknown Images That Tricked the Uncertainty Detector", fontsize=13, fontweight='bold', color='darkred')

for idx_plot, sample_idx in enumerate(worst_failures):
    img_tensor, _, orig_label = unknown_test_set[sample_idx]
    img = img_tensor.numpy().transpose((1, 2, 0)) * np.array(STD) + np.array(MEAN)
    img = np.clip(img, 0, 1)

    pred_name = KNOWN_NAMES[unknown_scores['preds'][sample_idx]]
    true_name = ALL_NAMES[orig_label]
    conf_pct  = unknown_scores['msp'][sample_idx] * 100

    axes[idx_plot].imshow(img)
    axes[idx_plot].set_title(f"True: {true_name.upper()}\nGuessed: {pred_name} ({conf_pct:.1f}%)", fontsize=10)
    axes[idx_plot].axis('off')

plt.tight_layout()
plt.show()